## Searching ahead of the driver

Everything so far anchored the search at the driver's current position, which answers the
question *what is around me*. Planning a trip asks a different one: **what will be around
me in half an hour** — the moment a break is due, a meal is planned, or the tank will need
filling. Knowing it upfront lets the driver decide where the pause will happen,
long before reaching the place, and pick a detour worth taking rather than the first exit
that comes up.

The demo answers it with a *minutes from position* setting: the search anchor is moved `n`
minutes **forward along the route**, so the list describes the neighborhood the car will
be in at that time. It composes naturally with detour reranking — a break is chosen among
the places that will be nearby *then*, ordered by how much of a detour each one costs.

This notebook computes that point from a single routing response, with no extra API call,
and renders it. The only ingredient needed is the route **spans**.

In [ ]:
from flexpolyline import decode

from here_search_demo.auth import Credentials
from here_search_demo.http import HTTPSession
from here_search_demo.routing_api import RoutingAPI

credentials = Credentials()
routing_api = RoutingAPI(credentials=credentials)
session = HTTPSession()

start_pos = 48.57436, 7.71556  # Strasbourg, where the trip started
at_pos = 48.03529, 7.37121  # near Colmar, where the driver is now
stop_pos = 45.75049, 4.84418  # Lyon, the destination

route = await routing_api.routes(
    session=session,
    origin=start_pos,
    destination=stop_pos,
    return_fields=("polyline", "summary"),
    spans=("duration", "dynamicSpeedInfo", "length"),
)
(section,) = RoutingAPI.sections(route)
waypoints = decode(section.polyline)

print(len(waypoints), "points,", len(section.spans), "spans")
section.spans[:2]

### Reading the spans

A span is a stretch of the route over which the requested attributes hold. Each of them
carries:

- `offset`: the index, in the decoded polyline, of the **first** point of the span
- `duration`: how many seconds it takes to drive it
- `length`: how many meters long it is

So span *i* covers the points from `spans[i]["offset"]` to `spans[i + 1]["offset"]`, the
last one running to the end of the polyline. Together they are a piecewise-constant speed
profile of the route, and their durations add up to the section duration.

`dynamicSpeedInfo` is what makes that profile exist. It is the attribute the API splits
the route on — a new span starts wherever the base or traffic speed changes. Asking for
`spans=("duration", "length")` alone returns **one** span covering the whole route, which
is useless here; `RouteEngine.route_spans` asks for all three for that reason.

Two operations are then enough to place a point *n* minutes ahead:

1. how many seconds separate the route start from the driver — `elapsed_sec_at_position()`
2. which point sits `n` minutes further — `position_at_x_sec_ahead()`

Both have to locate something *inside* a span, and a span offers two ways of doing it:

| handle | interpolation | cost | assumes |
| --- | --- | --- | --- |
| the point indices, `offset` to the next `offset` | `(index - start) / (end - start)` | a subtraction | points evenly spaced along the span |
| the `length`, in meters | walk the points, summing haversine distances until the target distance is reached | one haversine per point of the span | nothing |

They are not interchangeable, and each operation takes the one that fits its input and its
accuracy requirement.

`elapsed_sec_at_position()` starts from a *position*, which it first snaps to the nearest
polyline point — so what it holds is already an **index**, and index proration is the
natural continuation. It also does not need to be precise: the result feeds a search anchor
that is about to be moved tens of minutes away, and the snap itself already costs more
error than the proration does. Walking haversine distances here would buy nothing.

`position_at_x_sec_ahead()` has the opposite problem: it must *produce* a coordinate, and
the answer has to sit on the road. An index fraction would place it on a point of the
polyline chosen by rank rather than by distance — badly wrong wherever a span mixes a
straight motorway stretch, sampled sparsely, with a curve sampled densely. So it converts
its duration fraction into a **distance** through the span `length` and walks the geometry
for real, interpolating inside the final segment. This is the only place a true metric
distance is needed, and the only reason `haversine_m()` appears below.

That asymmetry is also what makes the round trip at the end of this notebook a genuine
check rather than a tautology: the two directions compute through different arithmetic, so
a mistake in one of them does not cancel out in the other.

In [ ]:
def elapsed_sec_at_position(spans, waypoints, lat, lon):
    """Return the seconds from the route start to the waypoint closest to (lat, lon)."""
    closest_index = min(
        range(len(waypoints)),
        key=lambda i: (lat - waypoints[i][0]) ** 2 + (lon - waypoints[i][1]) ** 2,
    )

    elapsed = 0.0
    last_index = len(waypoints) - 1
    for index, span in enumerate(spans):
        span_start = span["offset"]
        span_end = spans[index + 1]["offset"] if index + 1 < len(spans) else last_index
        if closest_index <= span_end:
            span_width = span_end - span_start
            if span_width > 0:  # prorate inside the span holding the position
                elapsed += span["duration"] * (closest_index - span_start) / span_width
            return elapsed
        elapsed += span["duration"]
    return elapsed


elapsed = elapsed_sec_at_position(section.spans, waypoints, *at_pos)
round(elapsed / 60), round(section.duration / 60)

The driver is that many minutes into a trip of that many minutes. Note that the position is
snapped to the closest polyline point with a plain squared-distance comparison: the search
anchor only has to be right to within a few meters, so neither map matching nor a
great-circle distance is warranted here.

The forward direction reuses the same span walk, stopped by a duration target rather than
by a point index, and finished with the metric handle rather than the index one. The span
holding the target gives the fraction of its duration to cover; the `length` turns that
fraction into meters; the points of the span are then walked until those meters are covered.

In [ ]:
import math

EARTH_RADIUS_M = 6_371_000.0


def haversine_m(lat1, lon1, lat2, lon2):
    """Great-circle distance in meters between two (lat, lon) points."""
    dlat, dlon = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2) ** 2
    )
    return EARTH_RADIUS_M * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def position_at_x_sec_ahead(spans, waypoints, elapsed, x_sec):
    """Return the (lat, lon) sitting *x_sec* seconds after *elapsed* along the route."""
    target = elapsed + x_sec
    accumulated = 0.0
    last_index = len(waypoints) - 1
    for index, span in enumerate(spans):
        span_duration = span.get("duration", 0)
        if accumulated + span_duration > target:
            fraction = (target - accumulated) / span_duration if span_duration > 0 else 0.0
            target_distance = span.get("length", 0) * fraction
            span_end = spans[index + 1]["offset"] if index + 1 < len(spans) else last_index
            points = waypoints[span["offset"] : span_end + 1]
            if len(points) < 2:
                return points[0] if points else waypoints[-1]
            covered = 0.0
            for (lat1, lon1), (lat2, lon2) in zip(points, points[1:]):
                leg = haversine_m(lat1, lon1, lat2, lon2)
                if covered + leg >= target_distance:  # interpolate inside the leg
                    ratio = (target_distance - covered) / leg if leg > 1e-9 else 0.0
                    return lat1 + (lat2 - lat1) * ratio, lon1 + (lon2 - lon1) * ratio
                covered += leg
            return points[-1]
        accumulated += span_duration
    return waypoints[-1]  # the target is past the destination


position_at_x_sec_ahead(section.spans, waypoints, elapsed, 20 * 60)

Both functions return a point that is *not* one of the polyline points, and both degrade
gracefully: a horizon longer than the remaining trip returns the destination, which is the
sensible answer — there is nothing to search beyond it.

The check is a round trip: measuring the elapsed time at the computed point must
give back the requested horizon.

In [ ]:
horizons_min = (5, 10, 20, 30, 45, 60, 600)

ahead = {}
for minutes in horizons_min:
    position = position_at_x_sec_ahead(section.spans, waypoints, elapsed, minutes * 60)
    ahead[minutes] = position
    measured = elapsed_sec_at_position(section.spans, waypoints, *position) - elapsed
    print(f"{minutes:>3} min ahead -> {position[0]:.5f}, {position[1]:.5f} "
          f"(measured back: {round(measured / 60)} min)")

The last row asks for ten hours ahead of a trip that is shorter than that: the destination
is returned and the measurement stops at the remaining duration.

The demo ships both functions in `widgets.route_geometry`:

In [ ]:
from here_search_demo.widgets.route_geometry import (
    elapsed_sec_at_position as demo_elapsed,
    position_at_x_sec_ahead as demo_position,
)

demo_elapsed(section.spans, waypoints, *at_pos) == elapsed and all(
    demo_position(section.spans, waypoints, elapsed, minutes * 60) == position
    for minutes, position in ahead.items()
)

### What it changes for the search

The anchor is the `latitude`/`longitude` pair of the search call; the corridor polyline
does not change. Moving the anchor twenty minutes forward re-orders — and largely renews —
the result list.

In [ ]:
from flexpolyline import encode

from here_search_demo.api import API
from here_search_demo.widgets.route_geometry import simplify_polyline

api = API(credentials=credentials)
search_polyline = encode(simplify_polyline(waypoints, 2000))


async def browse_at(position):
    """Browse restaurants along the route, anchored at *position*."""
    response = await api.browse(
        session=session,
        latitude=position[0],
        longitude=position[1],
        categories=["100-1000"],  # restaurants
        polyline=search_polyline,
        width=500,
        limit=20,
        lang="en",
    )
    return response.titles


here_and_now = await browse_at(at_pos)
in_20_min = await browse_at(ahead[20])

print("at the driver  :", here_and_now[:5])
print("20 min ahead   :", in_20_min[:5])
print("kept           :", len(set(here_and_now) & set(in_20_min)), "of", len(here_and_now))

Both lists are corridor-restricted and distance-ordered, but around two different centers:
the second one is the answer to *where could the break happen in twenty minutes*. Feeding
it to the [detour reranking](obm_12_detour_api.ipynb) is the natural next step. The
excursion is then measured from the future position too, so the list becomes the candidate
pauses ordered by their cost. The two features compose without any extra call, since the
same routing response provides the spans used here and the geometry used there.

### Seeing it

The horizons are drawn as markers along the route, from the driver's position onward.

In [ ]:
from ipyleaflet import AwesomeIcon, CircleMarker, Map, Marker, Polyline, TileLayer
from ipywidgets import HTML

def here_tile_layer():
    """Return a HERE Map Tile API v3 raster layer, as `widgets.input_map.PositionMap` does."""
    return TileLayer(
        url=(
            "https://maps.hereapi.com/v3/base/mc/{z}/{x}/{y}/png"
            f"?style=lite.day&ppi=400&size=512&apiKey={credentials.api_key}&lang=en"
        ),
        attribution="© HERE 2026",
    )


position_map = Map(
    center=at_pos,
    zoom=9,
    basemap={},
    layout={"height": "500px"},
    zoom_control=False,
    scroll_wheel_zoom=True,
    double_click_zoom=False,
    close_popup_on_click=False,
)
position_map.add(here_tile_layer())
position_map.add(Polyline(locations=waypoints, color="#2c7fb8", weight=3, fill=False))
position_map.add(Marker(location=at_pos, title="driver", draggable=False, icon=AwesomeIcon(name="car")))
position_map.add(Marker(location=stop_pos, title="stop", draggable=False, icon=AwesomeIcon(name="flag")))

for minutes in (5, 10, 20, 30, 45):
    marker = CircleMarker(
        location=ahead[minutes],
        radius=6,
        color="#e6550d",
        fill_color="#e6550d",
        fill_opacity=0.9,
        weight=1,
    )
    marker.popup = HTML(f"<b>in {minutes} min</b>")
    position_map.add(marker)

display(position_map)

The markers bunch up in the towns and spread out on the motorway: that is the speed profile
carried by the spans, and it is exactly why the anchor cannot be placed by distance. "In
thirty minutes" is a question a driver can answer; "in forty kilometers" is not.

### In the demo application

`RouteEngine` keeps the decoded waypoints and the spans of the current route, so the two
functions above are called on cached data:

- `RouteEngine.set_mins_from_pos(minutes)` stores the horizon and calls `_apply_mins_from_pos()`
- `_apply_mins_from_pos()` runs the two functions and stores the result as `future_position`
- `RouteEngine.search_at_position` returns `future_position or current_position`, and is
  what every search call uses as its anchor

On the widget side, `widgets.route.RouteController.build_mins_from_pos_text_w()` is the
text input, `RouteController.draw_future_position()` is the marker drawn above, and
`widgets.app` clears the result list when the horizon changes, since the previous results
were anchored elsewhere. The [application automation notebook](../obm_5_automate.ipynb) drives
all of it from code.

<!--
from urllib.parse import quote, unquote
svg = """<svg width="40" height="48" viewBox="0 0 40 48" fill="none" xmlns="http://www.w3.org/2000/svg"><path class="triangle" d="M11,36.8l-5.5,5.5L0,36.8H11z" fill="#48DAD0"></path><path class="HERE" d="M19.1,24.2c-1.2-1.4-1.1-2.1-0.4-2.8c0.9-0.9,1.7-0.5,2.7,0.5L19.1,24.2z M30.9,9.2c0.9-0.9,1.7-0.5,2.7,0.5
    L31.3,12C30.1,10.7,30.2,9.9,30.9,9.2z M38,11.6c-1.1,1.6-2.9,4.2-4.9,2.2l5-5c-0.4-0.5-0.8-0.9-1-1.1c-2.7-2.7-5.7-2.7-8-0.4
	c-1.6,1.6-2,3.4-1.5,5.1l-1.6-1.8c-0.5,0.3-2.4,1.9-0.9,4.5l-1.8-1.5l-2.4,2.4l3.2,3.2c-2.5-1.9-5.1-1.8-7.2,0.4
	c-2.3,2.3-2.1,5-0.4,7.3l-0.3-0.3c-2.3-2.3-4.7-1.5-5.9-0.3c-0.9,0.9-1.5,2.2-1.3,3.1L4,24.6l-2.6,2.6l9.6,9.6h5.2l-3.5-3.5
	c-1.8-1.8-1.8-2.8-1-3.7c0.8-0.8,1.8-0.3,3.6,1.4l3.4,3.4l2.6-2.6l-3.2-3.2c2.3,1.8,5.1,1.9,7.7-0.6l0,0c1.5-1.4,2-2.8,2-2.8
	l-1.9-1.3c-1.1,1.6-2.9,4.2-4.9,2.3l5-5l3.1,3.1l2.7-2.7l-3.9-3.9c-1.8-1.8-0.7-3.5,0-4.1c0.4,0.7,0.9,1.5,1.5,2.1
	c2.5,2.5,5.7,3,8.6,0.2l0,0c1.5-1.4,2-2.8,2-2.8S38,11.6,38,11.6z" fill="#000000"></path></svg>
"""
print(f"""![HERE](data:image/svg+xml,{quote(svg)})
<span style="float:right; width:90%;"><sub><b>Copyright (c) 2020-2026 HERE Global B.V. and its affiliate(s). All rights reserved.</b>
This software, including documentation, is protected by copyright controlled by HERE. All rights are reserved. Copying, including reproducing, 
storing, adapting or translating, any or all of this material requires the prior written consent of HERE. This material also contains confidential 
information which may not be disclosed to others without the prior written consent of HERE.</sub></span>""")
-->
![HERE](data:image/svg+xml,%3Csvg%20width%3D%2240%22%20height%3D%2248%22%20viewBox%3D%220%200%2040%2048%22%20fill%3D%22none%22%20xmlns%3D%22http%3A//www.w3.org/2000/svg%22%3E%3Cpath%20class%3D%22triangle%22%20d%3D%22M11%2C36.8l-5.5%2C5.5L0%2C36.8H11z%22%20fill%3D%22%2348DAD0%22%3E%3C/path%3E%3Cpath%20class%3D%22HERE%22%20d%3D%22M19.1%2C24.2c-1.2-1.4-1.1-2.1-0.4-2.8c0.9-0.9%2C1.7-0.5%2C2.7%2C0.5L19.1%2C24.2z%20M30.9%2C9.2c0.9-0.9%2C1.7-0.5%2C2.7%2C0.5%0A%20%20%20%20L31.3%2C12C30.1%2C10.7%2C30.2%2C9.9%2C30.9%2C9.2z%20M38%2C11.6c-1.1%2C1.6-2.9%2C4.2-4.9%2C2.2l5-5c-0.4-0.5-0.8-0.9-1-1.1c-2.7-2.7-5.7-2.7-8-0.4%0A%09c-1.6%2C1.6-2%2C3.4-1.5%2C5.1l-1.6-1.8c-0.5%2C0.3-2.4%2C1.9-0.9%2C4.5l-1.8-1.5l-2.4%2C2.4l3.2%2C3.2c-2.5-1.9-5.1-1.8-7.2%2C0.4%0A%09c-2.3%2C2.3-2.1%2C5-0.4%2C7.3l-0.3-0.3c-2.3-2.3-4.7-1.5-5.9-0.3c-0.9%2C0.9-1.5%2C2.2-1.3%2C3.1L4%2C24.6l-2.6%2C2.6l9.6%2C9.6h5.2l-3.5-3.5%0A%09c-1.8-1.8-1.8-2.8-1-3.7c0.8-0.8%2C1.8-0.3%2C3.6%2C1.4l3.4%2C3.4l2.6-2.6l-3.2-3.2c2.3%2C1.8%2C5.1%2C1.9%2C7.7-0.6l0%2C0c1.5-1.4%2C2-2.8%2C2-2.8%0A%09l-1.9-1.3c-1.1%2C1.6-2.9%2C4.2-4.9%2C2.3l5-5l3.1%2C3.1l2.7-2.7l-3.9-3.9c-1.8-1.8-0.7-3.5%2C0-4.1c0.4%2C0.7%2C0.9%2C1.5%2C1.5%2C2.1%0A%09c2.5%2C2.5%2C5.7%2C3%2C8.6%2C0.2l0%2C0c1.5-1.4%2C2-2.8%2C2-2.8S38%2C11.6%2C38%2C11.6z%22%20fill%3D%22%23000000%22%3E%3C/path%3E%3C/svg%3E%0A)
<span style="float:right; width:90%;"><sub><b>Copyright (c) 2020-2026 HERE Global B.V. and its affiliate(s). All rights reserved.</b>
This software, including documentation, is protected by copyright controlled by HERE. All rights are reserved. Copying, including reproducing, 
storing, adapting or translating, any or all of this material requires the prior written consent of HERE. This material also contains confidential 
information which may not be disclosed to others without the prior written consent of HERE.</sub></span>